# Preprocesamiento de datos de vinos y modelos — versión detallada para GitHub
**Adaptado a partir de tu archivo `preprocesamientoDeDatos.ipynb` (18 celdas originales).**

Conservamos tus bloques de exploración, duplicados, tipos, IQR, resumen, guardado, lectura y regresión lineal. Añadimos comentarios dentro del código, normalización, correlación y dos modelos por área.

**Qué queremos predecir:** `quality`, la puntuación de calidad del vino, a partir de 11 mediciones fisicoquímicas. La puntuación es ordinal; aquí la aproximamos mediante regresión numérica. Por eso las predicciones pueden tener decimales y usamos errores de regresión.

**Corrección de tu versión:** el capping se aplicaba a `data`, pero se guardaba `data_limpia`. Al releer este CSV, el modelo recibía datos sin ese capping. Ahora mantenemos claro qué tabla se usa en cada etapa.

**Orden:** preparar → cargar → explorar → deduplicar → revisar nulos/tipos → diagnosticar atípicos → guardar/leer → separar → correlación → tratamiento y normalización → modelos → métricas → comparación.

**Uso:** ejecutar las celdas de arriba hacia abajo. No depende de `src/vinos.py` ni de Google Drive. No se han incluido datos COVID.

## 0. Preparar los archivos y librerías
Esta es la única celda de configuración. En Colab descarga tu repositorio público. En tu computadora busca su carpeta. Si cambia librerías pide reiniciar antes de importarlas para evitar mezclar versiones de NumPy.

No elimina ni actualiza una carpeta de Colab que ya existe; para descargar una revisión nueva usa una sesión nueva o actualiza tu copia conscientemente.

In [ ]:
# Path permite trabajar con rutas sin escribir la ruta personal de cada compañero.
from pathlib import Path
import os, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

REPO_URL = "https://github.com/madahi-is/Machine-learning.git"
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Elegimos la ubicación según dónde se está ejecutando el notebook.
if EN_COLAB:
    destino = Path('/content/ml-vinos-covid')
    if not destino.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(destino)], check=True)
    # Cambiar a la raíz permite usar rutas relativas como data/raw/...
    os.chdir(destino)
else:
    inicio = Path.cwd().resolve()
    raiz = next((p for p in [inicio, *inicio.parents]
                 if (p / 'data/raw/winequality-red.csv').exists()), None)
    if raiz is None:
        raise FileNotFoundError('Abre el notebook dentro de tu proyecto Machine-learning.')
    os.chdir(raiz)

# Versiones utilizadas para comprobar esta entrega.
requeridos = {'numpy': '2.3.5', 'pandas': '2.2.3',
              'scikit-learn': '1.8.0', 'matplotlib': '3.10.8', 'seaborn': '0.13.2'}
pendientes = []
for paquete, esperado in requeridos.items():
    try:
        instalado = version(paquete)
    except PackageNotFoundError:
        instalado = None
    if instalado != esperado:
        pendientes.append(f'{paquete}=={esperado}')
# Solo instalamos cuando la versión requerida no está disponible.
if pendientes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pendientes], check=True)
    raise SystemExit('Paquetes instalados. Reinicia la sesión y vuelve a ejecutar esta celda antes de continuar.')
print('Proyecto:', Path.cwd())
print('Versiones:', {p: version(p) for p in requeridos})

## 1. Importar librerías
Conservamos tus cuatro librerías e indicamos para qué sirve cada una.

In [ ]:
# Importar librerias
import pandas as pd              # Manejar los datos como una tabla (DataFrame).
import numpy as np               # Operaciones numéricas, percentiles y recorte de valores.
import matplotlib.pyplot as plt  # Dibujar y guardar gráficos.
import seaborn as sns            # Representar la matriz de correlación como mapa de calor.
from IPython.display import display  # Mostrar tablas con formato dentro de Colab.

## 2. Lectura del dataset desde el proyecto
Sustituimos `drive.mount` y `/gdrive/MyDrive/...` por el CSV que ya está en GitHub. `data` permanecerá como copia original.

In [ ]:
# Lectura
# La ruta es relativa a la carpeta del proyecto, no al Drive de una persona.
ruta = Path('data/raw/winequality-red.csv')

# sep=None permite detectar si el archivo usa coma o punto y coma.
# engine='python' habilita esa detección automática del separador.
data = pd.read_csv(ruta, sep=None, engine='python')

# Quitamos espacios en los extremos de los encabezados, no de los valores.
data.columns = data.columns.str.strip()
print('Archivo leído:', ruta)

In [ ]:
# Comprobación rápida de la carga: shape muestra (filas, columnas).
print(data.shape)
# head() muestra cinco vinos para revisar que las columnas se separaron bien.
display(data.head())

## 3. Exploración general
Conservamos tu celda. La transposición `.T` coloca cada variable en una fila para facilitar la lectura.

In [ ]:
# Esto te mostrará los tipos de datos y las estadísticas generales.
print("Dimensiones del dataset:")
# shape muestra el número de registros y de columnas.
print(data.shape)

print("\nInformación del dataset:")
# info indica nombres, tipos y cantidad de valores no nulos.
data.info()

print("\nPrimeras filas:")
display(data.head())

print("\nEstadísticas descriptivas:")
# describe resume cantidad, media, dispersión, mínimo, cuartiles y máximo.
display(data.describe().T)

## 4. Inconsistencia: registros duplicados
Se consideran duplicadas las filas idénticas en todas las columnas. Las eliminamos para evitar que una copia aparezca en entrenamiento y otra en prueba. Sin identificador no podemos demostrar que toda coincidencia sea un error: documentamos esta decisión.

In [ ]:
# Inconsistencia: registros duplicados
# duplicated marca True en las repeticiones posteriores a la primera.
# sum cuenta esos True: cuántas filas se eliminarían.
duplicados = data.duplicated().sum()

print("Número de registros duplicados:")
print(duplicados)

In [ ]:
# Mostrar diez ejemplos de registros duplicados, como en tu notebook original.
# Esto permite inspeccionar las repeticiones antes de eliminarlas.
display(data[data.duplicated()].head(10))

## 5. Eliminar registros duplicados

In [ ]:
# Eliminar registros duplicados
# drop_duplicates conserva una copia de cada fila.
# copy crea una tabla independiente para mantener intacto el original data.
data_limpia = data.drop_duplicates().copy()

print("Registros antes de eliminar duplicados:", len(data))
print("Registros después de eliminar duplicados:", len(data_limpia))

In [ ]:
# Verificar que la operación anterior eliminó todas las repeticiones exactas.
print("Duplicados después de la limpieza:")
print(data_limpia.duplicated().sum())

## 6. Tipos de datos, valores faltantes y validaciones
Conservamos tu revisión de tipos y ampliamos las comprobaciones. No sustituimos automáticamente textos inesperados: primero hay que investigar su significado.

In [ ]:
# Tipos de datos
print(data_limpia.dtypes)

# Las columnas predictoras y quality deben ser numéricas para estos modelos.
if "quality" not in data_limpia.columns:
    raise ValueError("Falta la columna objetivo quality.")
if not all(pd.api.types.is_numeric_dtype(t) for t in data_limpia.dtypes):
    raise ValueError("Hay columnas no numéricas; revisa su contenido antes de continuar.")

In [ ]:
# isnull() detecta datos faltantes; sum() los cuenta por columna.
print('Valores faltantes por columna:')
print(data_limpia.isnull().sum())

# Un infinito no es un valor faltante normal: detenemos el análisis para revisarlo.
if np.isinf(data_limpia.to_numpy()).any():
    raise ValueError('Se detectaron valores infinitos; revisar el archivo.')

# No se inventa la respuesta de entrenamiento: se excluyen filas sin quality.
filas_sin_quality = int(data_limpia['quality'].isna().sum())
data_limpia = data_limpia.dropna(subset=['quality']).copy()
print('Filas excluidas por falta de quality:', filas_sin_quality)

# Comprobación de la escala esperada; no alteramos ni normalizamos quality.
if not data_limpia['quality'].between(0, 10).all():
    raise ValueError('Hay puntuaciones fuera de la escala esperada 0–10.')

# Si faltan predictores, se rellenarán más adelante con medianas de entrenamiento.
# En el archivo suministrado no hay valores faltantes.

## 7. Valores atípicos: diagnóstico con IQR
Mantenemos la lógica de tu bucle y la explicamos. Aquí **solo detectamos**, sin recortar todavía: los límites que use el modelo deben aprenderse después de separar entrenamiento/prueba.

Un valor extremo no necesariamente es un error. El capping conserva filas, pero modifica mediciones, por eso se registra como una decisión experimental. `quality` se excluye.

In [ ]:
# Valores atipicos
# Método de rango intercuartílico (IQR) para cada variable numérica.
# Corregimos data por data_limpia para analizar la tabla sin duplicados.
columnas_num = data_limpia.select_dtypes(include=[np.number]).columns
columnas_num = columnas_num.drop('quality')  # no se trata la variable objetivo como outlier

outliers_totales = 0
# Permite contar además cuántos vinos distintos tienen algún valor extremo.
filas_con_outliers = pd.Series(False, index=data_limpia.index)

for col in columnas_num:
    Q1 = data_limpia[col].quantile(0.25)  # El 25% de valores está por debajo de Q1.
    Q3 = data_limpia[col].quantile(0.75)  # El 75% de valores está por debajo de Q3.
    IQR = Q3 - Q1                       # Dispersión del 50% central de los datos.
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    # Marcamos los valores fuera del intervalo propuesto por la regla IQR.
    mascara = (data_limpia[col] < limite_inf) | (data_limpia[col] > limite_sup)
    n_outliers = int(mascara.sum())
    outliers_totales += n_outliers
    filas_con_outliers |= mascara
    print(f'{col}: {n_outliers} outliers (límites: {limite_inf:.3f} - {limite_sup:.3f})')

    # El capping de tu versión se aplica en la sección de entrenamiento.
    # No usamos estos límites globales para preparar los datos del modelo.

print(f'\nTotal de valores atípicos detectados: {outliers_totales}')
print('Vinos con al menos un valor atípico:', int(filas_con_outliers.sum()))
# No son la misma cantidad: un vino puede tener extremos en varias columnas.
print('En esta celda no se han modificado las mediciones.')

## 8. Resumen de limpieza inicial
Conservamos tu resumen y aclaramos que el recorte y la normalización aún no se aplicaron.

In [ ]:
# Resumen de deduplicación y de la revisión de faltantes.
print("  RESUMEN DE LIMPIEZA  ")

print("Registros originales:", len(data))
print("Registros después de la limpieza:", len(data_limpia))

print("\nValores faltantes:")
print(data_limpia.isnull().sum().sum())

print("\nRegistros duplicados:")
print(data_limpia.duplicated().sum())

print("\nNúmero de columnas:")
print(data_limpia.shape[1])

print("\nTipos de datos:")
print(data_limpia.dtypes)

print("Nota: este CSV todavía no incluye capping ni normalización.")

## 9. Guardar el dataset limpio
Conservamos tu operación de guardar un CSV. La salida está dentro del proyecto, sin rutas personales. En Colab estos archivos son temporales: hay que descargarlos si quieres conservarlos.

In [ ]:
# Guardar el dataset limpio
# Usamos results/ para las salidas, manteniendo intacto data/raw/.
ruta_salida = Path('results/vinos_detallado/winequality-red-limpio.csv')

# Obtener el directorio de la ruta de salida.
directorio_salida = ruta_salida.parent

# Crear el directorio si no existe; parents crea también las carpetas intermedias.
directorio_salida.mkdir(parents=True, exist_ok=True)

# index=False evita añadir una columna artificial con el índice de pandas.
# Guardamos la limpieza inicial; no un escalado aprendido con todo el dataset.
data_limpia.to_csv(ruta_salida, index=False)
print('Dataset limpio guardado en:')
print(ruta_salida)

## 10. Lectura del dataset limpio
Conservamos este paso para comprobar el archivo guardado. Lo llamamos `data_modelado` para no sobrescribir `data`, que sigue siendo el original.

In [ ]:
# Lectura del dataset limpio
# Ya no necesitamos montar Drive: el CSV está en la carpeta del proyecto.
data_modelado = pd.read_csv(ruta_salida)

# Comprobar que guardar y leer no cambió las dimensiones ni el contenido numérico.
assert data_modelado.shape == data_limpia.shape
assert np.allclose(data_modelado.to_numpy(), data_limpia.to_numpy(), equal_nan=True)
print(data_modelado.shape)
display(data_modelado.head())

## 11. Separar entrenamiento y prueba
El modelo aprende con el 80%. El 20% se reserva para la evaluación final. Esta división aleatoria supone observaciones independientes; no tenemos identificadores de lote para comprobar agrupaciones.

In [ ]:
from sklearn.model_selection import train_test_split

# Separar las características (X) de la variable objetivo (y)
# Usamos el CSV limpio, no el original con duplicados.
X = data_modelado.drop('quality', axis=1)
y = data_modelado['quality']

# Dividir el dataset en conjuntos de entrenamiento y prueba (80% entrenamiento, 20% prueba)
# random_state=42 permite repetir la misma partición.
# X contiene entradas; y contiene la puntuación que queremos predecir.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dimensiones del conjunto de entrenamiento:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("\nDimensiones del conjunto de prueba:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# No se puede aprender la mediana de una columna completamente vacía.
if X_train.isna().all().any():
    raise ValueError("Hay una columna vacía en entrenamiento; revisar antes de continuar.")

## 12. Matriz de correlación
Usamos únicamente entrenamiento para mantener la prueba reservada. Pearson mide relaciones lineales entre −1 y 1; no demuestra causalidad. Una correlación cercana a 0 no descarta una relación no lineal. No eliminamos variables automáticamente por esta matriz.

In [ ]:
# Unimos temporalmente las entradas y quality para incluir el objetivo en el gráfico.
# join alinea los registros por sus índices; no mezcla filas distintas.
datos_correlacion = X_train.join(y_train)
matriz_correlacion = datos_correlacion.corr(method='pearson')

# Crear un gráfico legible: números dentro de cada casilla y escala fija de -1 a 1.
fig_correlacion, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(matriz_correlacion, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, ax=ax, annot_kws={'fontsize': 8})
ax.set_title('Correlación — datos de entrenamiento antes del recorte')
plt.xticks(rotation=65, ha='right')
plt.yticks(rotation=0)
fig_correlacion.tight_layout()
plt.show()

## 13. Tratamiento de atípicos y normalización sin fuga de información
Conservamos tu **capping**: valores menores al límite inferior se sustituyen por ese límite y valores mayores al superior por el límite superior. No eliminamos filas ni cambiamos la puntuación objetivo.

`fit` aprende límites/medianas/mínimos/máximos de entrenamiento. `transform` los aplica sin recalcular. Durante validación cruzada necesitamos aprenderlos nuevamente en cada fold; por eso usamos `Pipeline`.

**Decisión previa:** `APLICAR_CAPPING=True`. No significa que sea universalmente mejor; no elegimos entre recortar o no recortar mirando el conjunto de prueba.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

APLICAR_CAPPING = True  # Decisión fijada antes de evaluar los modelos.

# Esta pequeña clase adapta tu cálculo de IQR al formato que necesita Pipeline.
# Los algoritmos de los modelos ya existen en scikit-learn; no los reimplementamos.
class RecorteIQR(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        # axis=0 calcula los percentiles por columna, no mezclando las variables.
        Q1, Q3 = np.nanpercentile(np.asarray(X), [25, 75], axis=0)
        IQR = Q3 - Q1
        # Guardamos los límites aprendidos para poder aplicarlos a datos nuevos.
        self.limite_inf_ = Q1 - 1.5 * IQR
        self.limite_sup_ = Q3 + 1.5 * IQR
        return self

    def transform(self, X):
        # Se aplica "capping" (recorte) en vez de eliminar, para no perder registros.
        # np.clip hace lo mismo que tus dos np.where: recorta al intervalo aprendido.
        return np.clip(np.asarray(X), self.limite_inf_, self.limite_sup_)

# La mediana se calcula solo en entrenamiento, si existen valores faltantes.
# Usamos una opción que desactiva el recorte sin eliminar la etapa del pipeline.
def nuevo_preprocesamiento():
    return [
        ('imputar', SimpleImputer(strategy='median')),
        ('recortar', RecorteIQR() if APLICAR_CAPPING else 'passthrough'),
        ('normalizar', MinMaxScaler()),
    ]
# Cada llamada crea objetos independientes para no compartir ajustes entre modelos.

### Ver qué cambia al recortar y normalizar
Esta celda permite inspeccionar el efecto antes de entrenar. MinMax calcula `(valor − mínimo_train) / (máximo_train − mínimo_train)`. Los árboles no necesitan escalado, pero aquí mantenemos el mismo flujo.

Estas matrices son ilustrativas: para validación se pasa `X_train` sin transformar a cada pipeline, para que cada fold aprenda sus propios parámetros. Un dato nuevo puede quedar fuera de [0,1]; no se recalcula el escalador con test.

In [ ]:
# 1. Aprender medianas solo con entrenamiento y rellenar nulos si los hay.
imputador_demo = SimpleImputer(strategy='median')
X_train_imputado = imputador_demo.fit_transform(X_train)

# 2. Aprender límites IQR con entrenamiento y aplicar tu recorte.
if APLICAR_CAPPING:
    recorte_demo = RecorteIQR()
    X_train_recortado = recorte_demo.fit_transform(X_train_imputado)
    # Contamos valores modificados por columna (no filas eliminadas).
    cambios = (X_train_imputado != X_train_recortado).sum(axis=0)
    display(pd.DataFrame({'variable': X.columns,
                          'limite_inf': recorte_demo.limite_inf_,
                          'limite_sup': recorte_demo.limite_sup_,
                          'valores_tratados': cambios}))
    print('Total de valores atípicos tratados en entrenamiento:', int(cambios.sum()))
else:
    X_train_recortado = X_train_imputado.copy()

# 3. Ajustar el escalador y transformar exclusivamente entrenamiento.
escalador_demo = MinMaxScaler()
X_train_normalizado = pd.DataFrame(
    escalador_demo.fit_transform(X_train_recortado),
    columns=X.columns, index=X_train.index
)
print('Primeras filas normalizadas:')
display(X_train_normalizado.head())
print('Número de registros conservados:', len(X_train_normalizado))
# No sobrescribimos X_train: los modelos siguientes harán su propio preprocesamiento.

## 14. Cómo evaluaremos los modelos
**MAE:** error absoluto promedio, en puntos de calidad. **MSE:** promedio del error al cuadrado. **RMSE:** raíz del MSE, también en puntos. Menor error es mejor. **R²:** medida del ajuste respecto a predecir la media del conjunto evaluado; puede ser negativo y no es porcentaje de aciertos.

No usamos accuracy porque predecimos una cantidad numérica. La regresión logística, pese a su nombre, se usa para clasificación.

Para detectar señales de sobreajuste comparamos entrenamiento y validación de los mismos cinco folds: un error mucho menor en entrenamiento puede indicar dificultad para generalizar. La brecha por sí sola no prueba sobreajuste ni existe un umbral universal. Si ambos errores son altos respecto a una referencia, podría haber poco ajuste o señal insuficiente.

Entrenamos y validamos en el 80%, elegimos por MAE de validación y solo después examinamos test. Cada modelo tendrá su celda de entrenamiento y otra de métricas, sin ocultarlas dentro de una función de evaluación.

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Dividir el conjunto de entrenamiento en cinco partes para validación cruzada.
# Cada vez se aprende con cuatro partes y se valida con la quinta.
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# scikit-learn devuelve errores con signo negativo para su criterio "mayor es mejor".
# Al mostrarlos invertimos el signo: nuestros MAE, MSE y RMSE serán positivos.
scoring = {'MAE': 'neg_mean_absolute_error', 'MSE': 'neg_mean_squared_error',
           'RMSE': 'neg_root_mean_squared_error', 'R2': 'r2'}

# Diccionarios: volver a ejecutar una celda sustituye su resultado, no lo duplica.
modelos = {}
resultados_cv = {}
resultados_test = {}
predicciones_test = pd.DataFrame({'quality_real': y_test})

## 15. Referencia: predecir siempre la media
Esta referencia ayuda a interpretar si los modelos aportan una mejora. Un resultado aislado es más difícil de valorar.

In [ ]:
from sklearn.dummy import DummyRegressor

# El modelo ignora las características y aprende la media de y del entrenamiento.
modelo_media = Pipeline(nuevo_preprocesamiento() + [('modelo', DummyRegressor())])
modelo_media.fit(X_train, y_train)
modelos['Media'] = modelo_media

# La media también se vuelve a aprender dentro de cada fold.
scores_media = cross_validate(modelo_media, X_train, y_train, cv=cv,
                             scoring=scoring, return_train_score=True, error_score='raise')
resultados_cv['Media'] = {'modelo': 'Media', 'area': 'Referencia',
    'MAE_train_CV': -scores_media['train_MAE'].mean(),
    'MAE_val_CV': -scores_media['test_MAE'].mean(),
    'MAE_val_std': scores_media['test_MAE'].std(),
    'MSE_val_CV': -scores_media['test_MSE'].mean(),
    'RMSE_val_CV': -scores_media['test_RMSE'].mean(),
    'R2_train_CV': scores_media['train_R2'].mean(),
    'R2_val_CV': scores_media['test_R2'].mean()}
display(pd.DataFrame([resultados_cv['Media']]).round(4))

# **Regresiones**

## Regresión lineal
Busca una combinación ponderada de las 11 variables para aproximar quality. Es regresión lineal múltiple porque utiliza varias entradas.

**Entrenamiento:** `.fit()` aprende del conjunto de entrenamiento. Las cinco predicciones de esta celda se muestran solo como ejemplo sobre entrenamiento; la prueba se consulta al final.

In [ ]:
from sklearn.linear_model import LinearRegression

# Tipo de modelo: Regresión lineal.
# Inicializar el modelo de regresión lineal.
# LinearRegression aprende un coeficiente para cada variable y un intercepto.
estimador_lineal = LinearRegression()

# Encadenar imputación → recorte IQR → normalización → modelo.
# Pipeline aprende cada transformación únicamente con los datos entregados a fit.
modelo_lineal = Pipeline(nuevo_preprocesamiento() + [('modelo', estimador_lineal)])

# Entrenar el modelo con los datos de entrenamiento (sin transformar previamente).
modelo_lineal.fit(X_train, y_train)
modelos['Lineal'] = modelo_lineal

# Realizar predicciones de ejemplo sobre entrenamiento, sin tocar todavía test.
# predict aplica los límites y el escalado aprendidos y después predice quality.
pred_train_lineal = modelo_lineal.predict(X_train)
print('Primeras 5 predicciones sobre entrenamiento (solo ilustración):')
for i in range(5):
    print(f'Real: {y_train.iloc[i]:.2f}, Predicho: {pred_train_lineal[i]:.2f}')

### Métricas de Regresión lineal: entrenamiento y validación
Los errores calculados sobre los datos usados para aprender son optimistas. La validación cruzada estima cómo se comporta el proceso en observaciones no usadas para ajustar ese fold.

In [ ]:
# Calcular el error cuadrático medio (MSE) en entrenamiento.
# Un error al cuadrado hace que las equivocaciones grandes pesen más.
mse_train_lineal = mean_squared_error(y_train, pred_train_lineal)

# Calcular el coeficiente de determinación (R-squared) en entrenamiento.
r2_train_lineal = r2_score(y_train, pred_train_lineal)
mae_train_lineal = mean_absolute_error(y_train, pred_train_lineal)
rmse_train_lineal = np.sqrt(mse_train_lineal)
print(f'MAE train: {mae_train_lineal:.4f} | MSE train: {mse_train_lineal:.4f}')
print(f'RMSE train: {rmse_train_lineal:.4f} | R² train: {r2_train_lineal:.4f}')

# Validar todo el pipeline: cada fold aprende sus propios límites IQR y escalador.
# Nunca se pasa X_train_normalizado aquí: hacerlo filtraría información entre folds.
# cross_validate crea copias para CV; no altera el modelo ajustado arriba.
scores_lineal = cross_validate(
    modelo_lineal, X_train, y_train, cv=cv, scoring=scoring,
    return_train_score=True, error_score='raise'
)

# El prefijo test de cross_validate significa VALIDACIÓN del fold, no X_test.
# Convertimos los errores negativos en positivos para interpretarlos normalmente.
mae_train_cv_lineal = -scores_lineal['train_MAE'].mean()
mae_val_cv_lineal = -scores_lineal['test_MAE'].mean()
resultados_cv['Lineal'] = {
    'modelo': 'Lineal', 'area': 'Regresiones',
    'MAE_train_CV': mae_train_cv_lineal, 'MAE_val_CV': mae_val_cv_lineal,
    'MAE_val_std': scores_lineal['test_MAE'].std(),
    'MSE_val_CV': -scores_lineal['test_MSE'].mean(),
    'RMSE_val_CV': -scores_lineal['test_RMSE'].mean(),
    'R2_train_CV': scores_lineal['train_R2'].mean(),
    'R2_val_CV': scores_lineal['test_R2'].mean()
}
display(pd.DataFrame([resultados_cv['Lineal']]).round(4))
print(f'Brecha MAE validación - entrenamiento por folds: {mae_val_cv_lineal - mae_train_cv_lineal:.4f}')
# Una brecha amplia es una señal que se interpreta junto al error y la referencia.
# No declaramos sobreajuste automáticamente por superar un número arbitrario.

## Ridge: regresión lineal regularizada
Añade una penalización L2 a los coeficientes grandes para reducir sensibilidad y posible sobreajuste. Puede ser útil con variables correlacionadas. Es una variante lineal.

**Entrenamiento:** `.fit()` aprende del conjunto de entrenamiento. Las cinco predicciones de esta celda se muestran solo como ejemplo sobre entrenamiento; la prueba se consulta al final.

In [ ]:
from sklearn.linear_model import Ridge

# Tipo de modelo: Ridge: regresión lineal regularizada.
# alpha controla la fuerza de regularización; 1.0 es una configuración inicial.
# Un alpha mayor penaliza más; no lo elegimos mirando los resultados de test.
estimador_ridge = Ridge(alpha=1.0)

# Encadenar imputación → recorte IQR → normalización → modelo.
# Pipeline aprende cada transformación únicamente con los datos entregados a fit.
modelo_ridge = Pipeline(nuevo_preprocesamiento() + [('modelo', estimador_ridge)])

# Entrenar el modelo con los datos de entrenamiento (sin transformar previamente).
modelo_ridge.fit(X_train, y_train)
modelos['Ridge'] = modelo_ridge

# Realizar predicciones de ejemplo sobre entrenamiento, sin tocar todavía test.
# predict aplica los límites y el escalado aprendidos y después predice quality.
pred_train_ridge = modelo_ridge.predict(X_train)
print('Primeras 5 predicciones sobre entrenamiento (solo ilustración):')
for i in range(5):
    print(f'Real: {y_train.iloc[i]:.2f}, Predicho: {pred_train_ridge[i]:.2f}')

### Métricas de Ridge: regresión lineal regularizada: entrenamiento y validación
Los errores calculados sobre los datos usados para aprender son optimistas. La validación cruzada estima cómo se comporta el proceso en observaciones no usadas para ajustar ese fold.

In [ ]:
# Calcular el error cuadrático medio (MSE) en entrenamiento.
# Un error al cuadrado hace que las equivocaciones grandes pesen más.
mse_train_ridge = mean_squared_error(y_train, pred_train_ridge)

# Calcular el coeficiente de determinación (R-squared) en entrenamiento.
r2_train_ridge = r2_score(y_train, pred_train_ridge)
mae_train_ridge = mean_absolute_error(y_train, pred_train_ridge)
rmse_train_ridge = np.sqrt(mse_train_ridge)
print(f'MAE train: {mae_train_ridge:.4f} | MSE train: {mse_train_ridge:.4f}')
print(f'RMSE train: {rmse_train_ridge:.4f} | R² train: {r2_train_ridge:.4f}')

# Validar todo el pipeline: cada fold aprende sus propios límites IQR y escalador.
# Nunca se pasa X_train_normalizado aquí: hacerlo filtraría información entre folds.
# cross_validate crea copias para CV; no altera el modelo ajustado arriba.
scores_ridge = cross_validate(
    modelo_ridge, X_train, y_train, cv=cv, scoring=scoring,
    return_train_score=True, error_score='raise'
)

# El prefijo test de cross_validate significa VALIDACIÓN del fold, no X_test.
# Convertimos los errores negativos en positivos para interpretarlos normalmente.
mae_train_cv_ridge = -scores_ridge['train_MAE'].mean()
mae_val_cv_ridge = -scores_ridge['test_MAE'].mean()
resultados_cv['Ridge'] = {
    'modelo': 'Ridge', 'area': 'Regresiones',
    'MAE_train_CV': mae_train_cv_ridge, 'MAE_val_CV': mae_val_cv_ridge,
    'MAE_val_std': scores_ridge['test_MAE'].std(),
    'MSE_val_CV': -scores_ridge['test_MSE'].mean(),
    'RMSE_val_CV': -scores_ridge['test_RMSE'].mean(),
    'R2_train_CV': scores_ridge['train_R2'].mean(),
    'R2_val_CV': scores_ridge['test_R2'].mean()
}
display(pd.DataFrame([resultados_cv['Ridge']]).round(4))
print(f'Brecha MAE validación - entrenamiento por folds: {mae_val_cv_ridge - mae_train_cv_ridge:.4f}')
# Una brecha amplia es una señal que se interpreta junto al error y la referencia.
# No declaramos sobreajuste automáticamente por superar un número arbitrario.

# **Árboles**

## Árbol de decisión para regresión
Divide los vinos mediante reglas sobre sus características. Cada hoja predice una puntuación a partir de los ejemplos que contiene. Puede representar relaciones no lineales.

**Entrenamiento:** `.fit()` aprende del conjunto de entrenamiento. Las cinco predicciones de esta celda se muestran solo como ejemplo sobre entrenamiento; la prueba se consulta al final.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Tipo de modelo: Árbol de decisión para regresión.
# max_depth=5 limita niveles para evitar un árbol excesivamente complejo.
# min_samples_leaf=5 exige al menos cinco ejemplos en cada hoja.
# random_state=42 permite repetir el ajuste.
estimador_arbol = DecisionTreeRegressor(max_depth=5, min_samples_leaf=5, random_state=42)

# Encadenar imputación → recorte IQR → normalización → modelo.
# Pipeline aprende cada transformación únicamente con los datos entregados a fit.
modelo_arbol = Pipeline(nuevo_preprocesamiento() + [('modelo', estimador_arbol)])

# Entrenar el modelo con los datos de entrenamiento (sin transformar previamente).
modelo_arbol.fit(X_train, y_train)
modelos['Arbol'] = modelo_arbol

# Realizar predicciones de ejemplo sobre entrenamiento, sin tocar todavía test.
# predict aplica los límites y el escalado aprendidos y después predice quality.
pred_train_arbol = modelo_arbol.predict(X_train)
print('Primeras 5 predicciones sobre entrenamiento (solo ilustración):')
for i in range(5):
    print(f'Real: {y_train.iloc[i]:.2f}, Predicho: {pred_train_arbol[i]:.2f}')

### Métricas de Árbol de decisión para regresión: entrenamiento y validación
Los errores calculados sobre los datos usados para aprender son optimistas. La validación cruzada estima cómo se comporta el proceso en observaciones no usadas para ajustar ese fold.

In [ ]:
# Calcular el error cuadrático medio (MSE) en entrenamiento.
# Un error al cuadrado hace que las equivocaciones grandes pesen más.
mse_train_arbol = mean_squared_error(y_train, pred_train_arbol)

# Calcular el coeficiente de determinación (R-squared) en entrenamiento.
r2_train_arbol = r2_score(y_train, pred_train_arbol)
mae_train_arbol = mean_absolute_error(y_train, pred_train_arbol)
rmse_train_arbol = np.sqrt(mse_train_arbol)
print(f'MAE train: {mae_train_arbol:.4f} | MSE train: {mse_train_arbol:.4f}')
print(f'RMSE train: {rmse_train_arbol:.4f} | R² train: {r2_train_arbol:.4f}')

# Validar todo el pipeline: cada fold aprende sus propios límites IQR y escalador.
# Nunca se pasa X_train_normalizado aquí: hacerlo filtraría información entre folds.
# cross_validate crea copias para CV; no altera el modelo ajustado arriba.
scores_arbol = cross_validate(
    modelo_arbol, X_train, y_train, cv=cv, scoring=scoring,
    return_train_score=True, error_score='raise'
)

# El prefijo test de cross_validate significa VALIDACIÓN del fold, no X_test.
# Convertimos los errores negativos en positivos para interpretarlos normalmente.
mae_train_cv_arbol = -scores_arbol['train_MAE'].mean()
mae_val_cv_arbol = -scores_arbol['test_MAE'].mean()
resultados_cv['Arbol'] = {
    'modelo': 'Arbol', 'area': 'Árboles',
    'MAE_train_CV': mae_train_cv_arbol, 'MAE_val_CV': mae_val_cv_arbol,
    'MAE_val_std': scores_arbol['test_MAE'].std(),
    'MSE_val_CV': -scores_arbol['test_MSE'].mean(),
    'RMSE_val_CV': -scores_arbol['test_RMSE'].mean(),
    'R2_train_CV': scores_arbol['train_R2'].mean(),
    'R2_val_CV': scores_arbol['test_R2'].mean()
}
display(pd.DataFrame([resultados_cv['Arbol']]).round(4))
print(f'Brecha MAE validación - entrenamiento por folds: {mae_val_cv_arbol - mae_train_cv_arbol:.4f}')
# Una brecha amplia es una señal que se interpreta junto al error y la referencia.
# No declaramos sobreajuste automáticamente por superar un número arbitrario.

## Random Forest para regresión
Entrena múltiples árboles y combina sus predicciones mediante un promedio. Cada árbol ve variaciones de los datos y de las características disponibles; esto suele reducir la variabilidad frente a un árbol aislado.

**Entrenamiento:** `.fit()` aprende del conjunto de entrenamiento. Las cinco predicciones de esta celda se muestran solo como ejemplo sobre entrenamiento; la prueba se consulta al final.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Tipo de modelo: Random Forest para regresión.
# n_estimators=200 construye doscientos árboles.
# min_samples_leaf=2 limita hojas muy pequeñas.
# n_jobs=1 limita el paralelismo para que sea fácil ejecutar en equipos modestos.
estimador_bosque = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=1)

# Encadenar imputación → recorte IQR → normalización → modelo.
# Pipeline aprende cada transformación únicamente con los datos entregados a fit.
modelo_bosque = Pipeline(nuevo_preprocesamiento() + [('modelo', estimador_bosque)])

# Entrenar el modelo con los datos de entrenamiento (sin transformar previamente).
modelo_bosque.fit(X_train, y_train)
modelos['RandomForest'] = modelo_bosque

# Realizar predicciones de ejemplo sobre entrenamiento, sin tocar todavía test.
# predict aplica los límites y el escalado aprendidos y después predice quality.
pred_train_bosque = modelo_bosque.predict(X_train)
print('Primeras 5 predicciones sobre entrenamiento (solo ilustración):')
for i in range(5):
    print(f'Real: {y_train.iloc[i]:.2f}, Predicho: {pred_train_bosque[i]:.2f}')

### Métricas de Random Forest para regresión: entrenamiento y validación
Los errores calculados sobre los datos usados para aprender son optimistas. La validación cruzada estima cómo se comporta el proceso en observaciones no usadas para ajustar ese fold.

In [ ]:
# Calcular el error cuadrático medio (MSE) en entrenamiento.
# Un error al cuadrado hace que las equivocaciones grandes pesen más.
mse_train_bosque = mean_squared_error(y_train, pred_train_bosque)

# Calcular el coeficiente de determinación (R-squared) en entrenamiento.
r2_train_bosque = r2_score(y_train, pred_train_bosque)
mae_train_bosque = mean_absolute_error(y_train, pred_train_bosque)
rmse_train_bosque = np.sqrt(mse_train_bosque)
print(f'MAE train: {mae_train_bosque:.4f} | MSE train: {mse_train_bosque:.4f}')
print(f'RMSE train: {rmse_train_bosque:.4f} | R² train: {r2_train_bosque:.4f}')

# Validar todo el pipeline: cada fold aprende sus propios límites IQR y escalador.
# Nunca se pasa X_train_normalizado aquí: hacerlo filtraría información entre folds.
# cross_validate crea copias para CV; no altera el modelo ajustado arriba.
scores_bosque = cross_validate(
    modelo_bosque, X_train, y_train, cv=cv, scoring=scoring,
    return_train_score=True, error_score='raise'
)

# El prefijo test de cross_validate significa VALIDACIÓN del fold, no X_test.
# Convertimos los errores negativos en positivos para interpretarlos normalmente.
mae_train_cv_bosque = -scores_bosque['train_MAE'].mean()
mae_val_cv_bosque = -scores_bosque['test_MAE'].mean()
resultados_cv['RandomForest'] = {
    'modelo': 'RandomForest', 'area': 'Árboles',
    'MAE_train_CV': mae_train_cv_bosque, 'MAE_val_CV': mae_val_cv_bosque,
    'MAE_val_std': scores_bosque['test_MAE'].std(),
    'MSE_val_CV': -scores_bosque['test_MSE'].mean(),
    'RMSE_val_CV': -scores_bosque['test_RMSE'].mean(),
    'R2_train_CV': scores_bosque['train_R2'].mean(),
    'R2_val_CV': scores_bosque['test_R2'].mean()
}
display(pd.DataFrame([resultados_cv['RandomForest']]).round(4))
print(f'Brecha MAE validación - entrenamiento por folds: {mae_val_cv_bosque - mae_train_cv_bosque:.4f}')
# Una brecha amplia es una señal que se interpreta junto al error y la referencia.
# No declaramos sobreajuste automáticamente por superar un número arbitrario.

# **SVM**

## SVR con kernel lineal
SVR es la versión de SVM para regresión. Busca una función con una franja de tolerancia alrededor de las respuestas. Con kernel lineal aproxima una relación lineal.

**Entrenamiento:** `.fit()` aprende del conjunto de entrenamiento. Las cinco predicciones de esta celda se muestran solo como ejemplo sobre entrenamiento; la prueba se consulta al final.

In [ ]:
from sklearn.svm import SVR

# Tipo de modelo: SVR con kernel lineal.
# kernel='linear' usa una relación lineal.
# C controla el compromiso entre complejidad y penalización de errores.
# epsilon=0.1 define la franja de tolerancia, en puntos de quality.
estimador_svr_lineal = SVR(kernel='linear', C=1.0, epsilon=0.1)

# Encadenar imputación → recorte IQR → normalización → modelo.
# Pipeline aprende cada transformación únicamente con los datos entregados a fit.
modelo_svr_lineal = Pipeline(nuevo_preprocesamiento() + [('modelo', estimador_svr_lineal)])

# Entrenar el modelo con los datos de entrenamiento (sin transformar previamente).
modelo_svr_lineal.fit(X_train, y_train)
modelos['SVR_lineal'] = modelo_svr_lineal

# Realizar predicciones de ejemplo sobre entrenamiento, sin tocar todavía test.
# predict aplica los límites y el escalado aprendidos y después predice quality.
pred_train_svr_lineal = modelo_svr_lineal.predict(X_train)
print('Primeras 5 predicciones sobre entrenamiento (solo ilustración):')
for i in range(5):
    print(f'Real: {y_train.iloc[i]:.2f}, Predicho: {pred_train_svr_lineal[i]:.2f}')

### Métricas de SVR con kernel lineal: entrenamiento y validación
Los errores calculados sobre los datos usados para aprender son optimistas. La validación cruzada estima cómo se comporta el proceso en observaciones no usadas para ajustar ese fold.

In [ ]:
# Calcular el error cuadrático medio (MSE) en entrenamiento.
# Un error al cuadrado hace que las equivocaciones grandes pesen más.
mse_train_svr_lineal = mean_squared_error(y_train, pred_train_svr_lineal)

# Calcular el coeficiente de determinación (R-squared) en entrenamiento.
r2_train_svr_lineal = r2_score(y_train, pred_train_svr_lineal)
mae_train_svr_lineal = mean_absolute_error(y_train, pred_train_svr_lineal)
rmse_train_svr_lineal = np.sqrt(mse_train_svr_lineal)
print(f'MAE train: {mae_train_svr_lineal:.4f} | MSE train: {mse_train_svr_lineal:.4f}')
print(f'RMSE train: {rmse_train_svr_lineal:.4f} | R² train: {r2_train_svr_lineal:.4f}')

# Validar todo el pipeline: cada fold aprende sus propios límites IQR y escalador.
# Nunca se pasa X_train_normalizado aquí: hacerlo filtraría información entre folds.
# cross_validate crea copias para CV; no altera el modelo ajustado arriba.
scores_svr_lineal = cross_validate(
    modelo_svr_lineal, X_train, y_train, cv=cv, scoring=scoring,
    return_train_score=True, error_score='raise'
)

# El prefijo test de cross_validate significa VALIDACIÓN del fold, no X_test.
# Convertimos los errores negativos en positivos para interpretarlos normalmente.
mae_train_cv_svr_lineal = -scores_svr_lineal['train_MAE'].mean()
mae_val_cv_svr_lineal = -scores_svr_lineal['test_MAE'].mean()
resultados_cv['SVR_lineal'] = {
    'modelo': 'SVR_lineal', 'area': 'SVM',
    'MAE_train_CV': mae_train_cv_svr_lineal, 'MAE_val_CV': mae_val_cv_svr_lineal,
    'MAE_val_std': scores_svr_lineal['test_MAE'].std(),
    'MSE_val_CV': -scores_svr_lineal['test_MSE'].mean(),
    'RMSE_val_CV': -scores_svr_lineal['test_RMSE'].mean(),
    'R2_train_CV': scores_svr_lineal['train_R2'].mean(),
    'R2_val_CV': scores_svr_lineal['test_R2'].mean()
}
display(pd.DataFrame([resultados_cv['SVR_lineal']]).round(4))
print(f'Brecha MAE validación - entrenamiento por folds: {mae_val_cv_svr_lineal - mae_train_cv_svr_lineal:.4f}')
# Una brecha amplia es una señal que se interpreta junto al error y la referencia.
# No declaramos sobreajuste automáticamente por superar un número arbitrario.

## SVR con kernel RBF
El kernel RBF permite relaciones no lineales. Su flexibilidad puede ayudar, pero también puede aumentar el sobreajuste. Necesita características en escalas comparables.

**Entrenamiento:** `.fit()` aprende del conjunto de entrenamiento. Las cinco predicciones de esta celda se muestran solo como ejemplo sobre entrenamiento; la prueba se consulta al final.

In [ ]:
from sklearn.svm import SVR

# Tipo de modelo: SVR con kernel RBF.
# kernel='rbf' permite una función no lineal.
# C=10.0 penaliza las desviaciones; no garantiza mejores resultados.
# gamma='scale' calcula el alcance del kernel usando las entradas de entrenamiento.
estimador_svr_rbf = SVR(kernel='rbf', C=10.0, epsilon=0.1, gamma='scale')

# Encadenar imputación → recorte IQR → normalización → modelo.
# Pipeline aprende cada transformación únicamente con los datos entregados a fit.
modelo_svr_rbf = Pipeline(nuevo_preprocesamiento() + [('modelo', estimador_svr_rbf)])

# Entrenar el modelo con los datos de entrenamiento (sin transformar previamente).
modelo_svr_rbf.fit(X_train, y_train)
modelos['SVR_RBF'] = modelo_svr_rbf

# Realizar predicciones de ejemplo sobre entrenamiento, sin tocar todavía test.
# predict aplica los límites y el escalado aprendidos y después predice quality.
pred_train_svr_rbf = modelo_svr_rbf.predict(X_train)
print('Primeras 5 predicciones sobre entrenamiento (solo ilustración):')
for i in range(5):
    print(f'Real: {y_train.iloc[i]:.2f}, Predicho: {pred_train_svr_rbf[i]:.2f}')

### Métricas de SVR con kernel RBF: entrenamiento y validación
Los errores calculados sobre los datos usados para aprender son optimistas. La validación cruzada estima cómo se comporta el proceso en observaciones no usadas para ajustar ese fold.

In [ ]:
# Calcular el error cuadrático medio (MSE) en entrenamiento.
# Un error al cuadrado hace que las equivocaciones grandes pesen más.
mse_train_svr_rbf = mean_squared_error(y_train, pred_train_svr_rbf)

# Calcular el coeficiente de determinación (R-squared) en entrenamiento.
r2_train_svr_rbf = r2_score(y_train, pred_train_svr_rbf)
mae_train_svr_rbf = mean_absolute_error(y_train, pred_train_svr_rbf)
rmse_train_svr_rbf = np.sqrt(mse_train_svr_rbf)
print(f'MAE train: {mae_train_svr_rbf:.4f} | MSE train: {mse_train_svr_rbf:.4f}')
print(f'RMSE train: {rmse_train_svr_rbf:.4f} | R² train: {r2_train_svr_rbf:.4f}')

# Validar todo el pipeline: cada fold aprende sus propios límites IQR y escalador.
# Nunca se pasa X_train_normalizado aquí: hacerlo filtraría información entre folds.
# cross_validate crea copias para CV; no altera el modelo ajustado arriba.
scores_svr_rbf = cross_validate(
    modelo_svr_rbf, X_train, y_train, cv=cv, scoring=scoring,
    return_train_score=True, error_score='raise'
)

# El prefijo test de cross_validate significa VALIDACIÓN del fold, no X_test.
# Convertimos los errores negativos en positivos para interpretarlos normalmente.
mae_train_cv_svr_rbf = -scores_svr_rbf['train_MAE'].mean()
mae_val_cv_svr_rbf = -scores_svr_rbf['test_MAE'].mean()
resultados_cv['SVR_RBF'] = {
    'modelo': 'SVR_RBF', 'area': 'SVM',
    'MAE_train_CV': mae_train_cv_svr_rbf, 'MAE_val_CV': mae_val_cv_svr_rbf,
    'MAE_val_std': scores_svr_rbf['test_MAE'].std(),
    'MSE_val_CV': -scores_svr_rbf['test_MSE'].mean(),
    'RMSE_val_CV': -scores_svr_rbf['test_RMSE'].mean(),
    'R2_train_CV': scores_svr_rbf['train_R2'].mean(),
    'R2_val_CV': scores_svr_rbf['test_R2'].mean()
}
display(pd.DataFrame([resultados_cv['SVR_RBF']]).round(4))
print(f'Brecha MAE validación - entrenamiento por folds: {mae_val_cv_svr_rbf - mae_train_cv_svr_rbf:.4f}')
# Una brecha amplia es una señal que se interpreta junto al error y la referencia.
# No declaramos sobreajuste automáticamente por superar un número arbitrario.

# **Comparación y elección del modelo**
Comparamos dos modelos por área usando exactamente la misma partición y folds. Elegimos por menor MAE de validación, antes de consultar test. La desviación estándar entre folds describe variación; no es un intervalo de confianza.

Los hiperparámetros son valores iniciales: no se ha hecho búsqueda exhaustiva. Para mejorarlos en otra etapa usaríamos validación dentro de entrenamiento, sin optimizar sobre test.

In [ ]:
# Comprobar que no falta ejecutar ningún modelo o la referencia.
if len(resultados_cv) != 7:
    raise RuntimeError('Ejecuta primero los seis modelos y la referencia de la media.')

# Una fila por modelo. Menor MAE de validación se muestra primero.
comparacion_cv = pd.DataFrame(resultados_cv.values()).sort_values('MAE_val_CV')
comparacion_cv['brecha_MAE'] = comparacion_cv['MAE_val_CV'] - comparacion_cv['MAE_train_CV']
display(comparacion_cv.round(4))

# Seleccionar entre los seis modelos; la media es únicamente una referencia.
candidatos = comparacion_cv[comparacion_cv['area'] != 'Referencia']
mejor_nombre = candidatos.iloc[0]['modelo']
print('Modelo elegido por validación:', mejor_nombre)
print('Mejor modelo de cada área:')
display(candidatos.groupby('area', sort=False).head(1)[['area', 'modelo', 'MAE_val_CV']])

# **Evaluación final sobre el conjunto de prueba**
Ahora conservamos tu estilo original: `predict(X_test)`, mostrar cinco predicciones y calcular métricas en otra celda, para cada modelo. Los modelos ya fueron fijados y la elección se hizo con validación. No ajustamos parámetros a partir de estos resultados.

**Limitación del trabajo:** ya vimos este test en versiones anteriores del proyecto. No debe presentarse como una confirmación completamente nueva e independiente; una evaluación estricta adicional requeriría datos no explorados previamente.

## Regresión lineal: predicciones sobre prueba

In [ ]:
# Realizar predicciones sobre el conjunto de prueba.
# No se llama fit: el modelo aplica lo aprendido con entrenamiento.
y_pred_lineal = modelo_lineal.predict(X_test)
predicciones_test['Lineal'] = y_pred_lineal

print('Primeras 5 predicciones del modelo — prueba:')
for i in range(5):
    print(f'Real: {y_test.iloc[i]:.2f}, Predicho: {y_pred_lineal[i]:.2f}')

### Métricas finales de Regresión lineal

In [ ]:
# Calcular el error cuadrático medio (MSE), como en tu notebook original.
mse_lineal = mean_squared_error(y_test, y_pred_lineal)

# Calcular el coeficiente de determinación (R-squared).
r2_lineal = r2_score(y_test, y_pred_lineal)

# Añadimos MAE y RMSE para expresar los errores en puntos de calidad.
mae_lineal = mean_absolute_error(y_test, y_pred_lineal)
rmse_lineal = np.sqrt(mse_lineal)
print(f'Mean Squared Error (MSE): {mse_lineal:.4f}')
print(f'R-squared (R2): {r2_lineal:.4f}')
print(f'Mean Absolute Error (MAE): {mae_lineal:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_lineal:.4f}')

# Guardar para comparar al final sin repetir cálculos ni duplicar filas.
resultados_test['Lineal'] = {'modelo': 'Lineal', 'area': 'Regresiones',
    'MAE': mae_lineal, 'MSE': mse_lineal, 'RMSE': rmse_lineal, 'R2': r2_lineal}

## Ridge: regresión lineal regularizada: predicciones sobre prueba

In [ ]:
# Realizar predicciones sobre el conjunto de prueba.
# No se llama fit: el modelo aplica lo aprendido con entrenamiento.
y_pred_ridge = modelo_ridge.predict(X_test)
predicciones_test['Ridge'] = y_pred_ridge

print('Primeras 5 predicciones del modelo — prueba:')
for i in range(5):
    print(f'Real: {y_test.iloc[i]:.2f}, Predicho: {y_pred_ridge[i]:.2f}')

### Métricas finales de Ridge: regresión lineal regularizada

In [ ]:
# Calcular el error cuadrático medio (MSE), como en tu notebook original.
mse_ridge = mean_squared_error(y_test, y_pred_ridge)

# Calcular el coeficiente de determinación (R-squared).
r2_ridge = r2_score(y_test, y_pred_ridge)

# Añadimos MAE y RMSE para expresar los errores en puntos de calidad.
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mse_ridge)
print(f'Mean Squared Error (MSE): {mse_ridge:.4f}')
print(f'R-squared (R2): {r2_ridge:.4f}')
print(f'Mean Absolute Error (MAE): {mae_ridge:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_ridge:.4f}')

# Guardar para comparar al final sin repetir cálculos ni duplicar filas.
resultados_test['Ridge'] = {'modelo': 'Ridge', 'area': 'Regresiones',
    'MAE': mae_ridge, 'MSE': mse_ridge, 'RMSE': rmse_ridge, 'R2': r2_ridge}

## Árbol de decisión para regresión: predicciones sobre prueba

In [ ]:
# Realizar predicciones sobre el conjunto de prueba.
# No se llama fit: el modelo aplica lo aprendido con entrenamiento.
y_pred_arbol = modelo_arbol.predict(X_test)
predicciones_test['Arbol'] = y_pred_arbol

print('Primeras 5 predicciones del modelo — prueba:')
for i in range(5):
    print(f'Real: {y_test.iloc[i]:.2f}, Predicho: {y_pred_arbol[i]:.2f}')

### Métricas finales de Árbol de decisión para regresión

In [ ]:
# Calcular el error cuadrático medio (MSE), como en tu notebook original.
mse_arbol = mean_squared_error(y_test, y_pred_arbol)

# Calcular el coeficiente de determinación (R-squared).
r2_arbol = r2_score(y_test, y_pred_arbol)

# Añadimos MAE y RMSE para expresar los errores en puntos de calidad.
mae_arbol = mean_absolute_error(y_test, y_pred_arbol)
rmse_arbol = np.sqrt(mse_arbol)
print(f'Mean Squared Error (MSE): {mse_arbol:.4f}')
print(f'R-squared (R2): {r2_arbol:.4f}')
print(f'Mean Absolute Error (MAE): {mae_arbol:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_arbol:.4f}')

# Guardar para comparar al final sin repetir cálculos ni duplicar filas.
resultados_test['Arbol'] = {'modelo': 'Arbol', 'area': 'Árboles',
    'MAE': mae_arbol, 'MSE': mse_arbol, 'RMSE': rmse_arbol, 'R2': r2_arbol}

## Random Forest para regresión: predicciones sobre prueba

In [ ]:
# Realizar predicciones sobre el conjunto de prueba.
# No se llama fit: el modelo aplica lo aprendido con entrenamiento.
y_pred_bosque = modelo_bosque.predict(X_test)
predicciones_test['RandomForest'] = y_pred_bosque

print('Primeras 5 predicciones del modelo — prueba:')
for i in range(5):
    print(f'Real: {y_test.iloc[i]:.2f}, Predicho: {y_pred_bosque[i]:.2f}')

### Métricas finales de Random Forest para regresión

In [ ]:
# Calcular el error cuadrático medio (MSE), como en tu notebook original.
mse_bosque = mean_squared_error(y_test, y_pred_bosque)

# Calcular el coeficiente de determinación (R-squared).
r2_bosque = r2_score(y_test, y_pred_bosque)

# Añadimos MAE y RMSE para expresar los errores en puntos de calidad.
mae_bosque = mean_absolute_error(y_test, y_pred_bosque)
rmse_bosque = np.sqrt(mse_bosque)
print(f'Mean Squared Error (MSE): {mse_bosque:.4f}')
print(f'R-squared (R2): {r2_bosque:.4f}')
print(f'Mean Absolute Error (MAE): {mae_bosque:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_bosque:.4f}')

# Guardar para comparar al final sin repetir cálculos ni duplicar filas.
resultados_test['RandomForest'] = {'modelo': 'RandomForest', 'area': 'Árboles',
    'MAE': mae_bosque, 'MSE': mse_bosque, 'RMSE': rmse_bosque, 'R2': r2_bosque}

## SVR con kernel lineal: predicciones sobre prueba

In [ ]:
# Realizar predicciones sobre el conjunto de prueba.
# No se llama fit: el modelo aplica lo aprendido con entrenamiento.
y_pred_svr_lineal = modelo_svr_lineal.predict(X_test)
predicciones_test['SVR_lineal'] = y_pred_svr_lineal

print('Primeras 5 predicciones del modelo — prueba:')
for i in range(5):
    print(f'Real: {y_test.iloc[i]:.2f}, Predicho: {y_pred_svr_lineal[i]:.2f}')

### Métricas finales de SVR con kernel lineal

In [ ]:
# Calcular el error cuadrático medio (MSE), como en tu notebook original.
mse_svr_lineal = mean_squared_error(y_test, y_pred_svr_lineal)

# Calcular el coeficiente de determinación (R-squared).
r2_svr_lineal = r2_score(y_test, y_pred_svr_lineal)

# Añadimos MAE y RMSE para expresar los errores en puntos de calidad.
mae_svr_lineal = mean_absolute_error(y_test, y_pred_svr_lineal)
rmse_svr_lineal = np.sqrt(mse_svr_lineal)
print(f'Mean Squared Error (MSE): {mse_svr_lineal:.4f}')
print(f'R-squared (R2): {r2_svr_lineal:.4f}')
print(f'Mean Absolute Error (MAE): {mae_svr_lineal:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_svr_lineal:.4f}')

# Guardar para comparar al final sin repetir cálculos ni duplicar filas.
resultados_test['SVR_lineal'] = {'modelo': 'SVR_lineal', 'area': 'SVM',
    'MAE': mae_svr_lineal, 'MSE': mse_svr_lineal, 'RMSE': rmse_svr_lineal, 'R2': r2_svr_lineal}

## SVR con kernel RBF: predicciones sobre prueba

In [ ]:
# Realizar predicciones sobre el conjunto de prueba.
# No se llama fit: el modelo aplica lo aprendido con entrenamiento.
y_pred_svr_rbf = modelo_svr_rbf.predict(X_test)
predicciones_test['SVR_RBF'] = y_pred_svr_rbf

print('Primeras 5 predicciones del modelo — prueba:')
for i in range(5):
    print(f'Real: {y_test.iloc[i]:.2f}, Predicho: {y_pred_svr_rbf[i]:.2f}')

### Métricas finales de SVR con kernel RBF

In [ ]:
# Calcular el error cuadrático medio (MSE), como en tu notebook original.
mse_svr_rbf = mean_squared_error(y_test, y_pred_svr_rbf)

# Calcular el coeficiente de determinación (R-squared).
r2_svr_rbf = r2_score(y_test, y_pred_svr_rbf)

# Añadimos MAE y RMSE para expresar los errores en puntos de calidad.
mae_svr_rbf = mean_absolute_error(y_test, y_pred_svr_rbf)
rmse_svr_rbf = np.sqrt(mse_svr_rbf)
print(f'Mean Squared Error (MSE): {mse_svr_rbf:.4f}')
print(f'R-squared (R2): {r2_svr_rbf:.4f}')
print(f'Mean Absolute Error (MAE): {mae_svr_rbf:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_svr_rbf:.4f}')

# Guardar para comparar al final sin repetir cálculos ni duplicar filas.
resultados_test['SVR_RBF'] = {'modelo': 'SVR_RBF', 'area': 'SVM',
    'MAE': mae_svr_rbf, 'MSE': mse_svr_rbf, 'RMSE': rmse_svr_rbf, 'R2': r2_svr_rbf}

## Comparación final y referencia
La tabla permite informar resultados de todos los modelos, pero no sustituye el criterio de selección fijado con CV.

In [ ]:
# Evaluar también la referencia con el mismo conjunto de prueba.
pred_media = modelo_media.predict(X_test)
predicciones_test['Media'] = pred_media
resultados_test['Media'] = {'modelo': 'Media', 'area': 'Referencia',
    'MAE': mean_absolute_error(y_test, pred_media),
    'MSE': mean_squared_error(y_test, pred_media),
    'RMSE': np.sqrt(mean_squared_error(y_test, pred_media)),
    'R2': r2_score(y_test, pred_media)}
if len(resultados_test) != 7:
    raise RuntimeError('Falta ejecutar alguna evaluación final.')
comparacion_test = pd.DataFrame(resultados_test.values())
comparacion_test['seleccionado_por_cv'] = comparacion_test['modelo'] == mejor_nombre
display(comparacion_test.round(4))
print('La selección sigue siendo:', mejor_nombre)

## Guardar resultados del experimento
Se guardan las tablas, la figura y las versiones. El CSV inicial sin duplicados es distinto a las matrices preparadas por el modelo. No vuelvas a usar matrices ya normalizadas como entrada a validación cruzada.

In [ ]:
import json, platform

# Reutilizamos la carpeta creada al guardar la limpieza inicial.
comparacion_cv.to_csv(directorio_salida / 'comparacion_cv.csv', index=False)
comparacion_test.to_csv(directorio_salida / 'comparacion_test.csv', index=False)
predicciones_test.to_csv(directorio_salida / 'predicciones_test.csv', index_label='fila_csv_limpio')
matriz_correlacion.to_csv(directorio_salida / 'correlacion_train.csv')
fig_correlacion.savefig(directorio_salida / 'correlacion_train.png', dpi=150, bbox_inches='tight')

# Exportar el preprocesamiento aprendido por el modelo seleccionado.
# [:-1] toma todas las etapas menos el estimador. transform nunca reajusta.
pre_final = modelos[mejor_nombre][:-1]
for nombre, tabla in [('train', X_train), ('test', X_test)]:
    preparada = pd.DataFrame(pre_final.transform(tabla), columns=X.columns, index=tabla.index)
    preparada.to_csv(directorio_salida / f'X_{nombre}_normalizado.csv', index_label='fila_csv_limpio')

# Registrar qué filas pertenecen a cada partición, incluido el índice del original.
pd.DataFrame({'fila_csv_limpio': data_modelado.index,
              'fila_csv_original': data_limpia.index,
              'particion': ['train' if i in X_train.index else 'test' for i in data_modelado.index]}
             ).to_csv(directorio_salida / 'particiones.csv', index=False)
resumen = {'filas_originales': len(data), 'duplicados_eliminados': int(duplicados),
           'filas_sin_quality': filas_sin_quality, 'filas_limpias': len(data_limpia),
           'filas_train': len(X_train), 'filas_test': len(X_test),
           'capping': APLICAR_CAPPING, 'semilla': 42, 'modelo_elegido_cv': mejor_nombre,
           'python': platform.python_version(), 'versiones': {p: version(p) for p in requeridos}}
(directorio_salida / 'resumen.json').write_text(
    json.dumps(resumen, indent=2, ensure_ascii=False), encoding='utf-8'
)
print('Archivos guardados en:', directorio_salida.resolve())

## Interpretación para el informe
1. Explicar 1599 filas originales, 240 duplicados exactos y 1359 filas sin duplicados en el archivo entregado.
2. Distinguir valores atípicos de filas afectadas. Justificar que el recorte conserva filas, pero modifica mediciones.
3. Explicar por qué el escalador, los límites IQR y la mediana se aprenden con entrenamiento.
4. Comparar el mejor MAE de validación de cada área y contrastarlo con la referencia.
5. Examinar la brecha entre entrenamiento y validación sin declarar sobreajuste por una regla arbitraria.
6. Informar las métricas finales y sus límites: tamaño del dataset, objetivo ordinal, posible agrupación de muestras y test explorado previamente.

Referencias de consulta: [evitar fuga de información](https://scikit-learn.org/stable/common_pitfalls.html) y [métricas de evaluación](https://scikit-learn.org/stable/modules/model_evaluation.html).

**Publicar este notebook:** reemplazar `notebooks/01_vinos.ipynb` en `madahi-is/Machine-learning`. Ejecutar en Colab no guarda automáticamente en GitHub. También puedes usar Archivo → Guardar una copia en GitHub y seleccionar esa ruta.

Los resultados del script anterior `src/vinos.py` pueden diferir: allí el recorte está desactivado por defecto. Este notebook es autocontenido y lo activa explícitamente.